### Code to avoid sudden kill when working with sklearn and torch simultaniuosly

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

### Libraries

In [18]:
# stantard libs
import numpy as np
import pandas as pd

# mac optimized ML and vectorized computing
import mlx.core as mx
import mlx.nn as mnn
import mlx.optimizers as optim

# plotting
import plotly.graph_objects as go

# dataset
from sklearn.datasets import load_iris

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# torch
import torch
import torch.nn as nn
import torch.optim as toptim

### Perceptron Classification through binary partiction of R<sup>3</sup> space

In [19]:
# load data (Setosa = 0, Versicolor = 1)
iris = load_iris(as_frame=True)
df = iris.frame
binary_df = df[df['target'] < 2]

# features for 3D plotting
X_3d = binary_df.iloc[:, [0, 1, 2]].values  # Sepal Length, Sepal Width, Petal Length
y_binary = binary_df['target'].values

# train
clf = Perceptron(max_iter=1000, random_state=42)
clf.fit(X_3d, y_binary)

# extract weights and bias to calculate  hyperplane
w = clf.coef_[0]
b = clf.intercept_[0]

# 3. Create a meshgrid for the plane
x_min, x_max = X_3d[:, 0].min(), X_3d[:, 0].max()
y_min, y_max = X_3d[:, 1].min(), X_3d[:, 1].max()

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 10),
                     np.linspace(y_min, y_max, 10))

# calculate  z values for  hyperplane
# w[0]*x + w[1]*y + w[2]*z + b = 0  =>  z = -(w[0]*x + w[1]*y + b) / w[2]
zz = -(w[0] * xx + w[1] * yy + b) / w[2]

# moving plot
fig = go.Figure()

# Setosa points
fig.add_trace(go.Scatter3d(
    x=X_3d[y_binary == 0, 0],
    y=X_3d[y_binary == 0, 1],
    z=X_3d[y_binary == 0, 2],
    mode='markers',
    marker=dict(size=6, color='pink', opacity=0.8),
    name='Setosa'
))

# Versicolor points
fig.add_trace(go.Scatter3d(
    x=X_3d[y_binary == 1, 0],
    y=X_3d[y_binary == 1, 1],
    z=X_3d[y_binary == 1, 2],
    mode='markers',
    marker=dict(size=6, color='green', opacity=0.8),
    name='Versicolor'
))

# hyperplane
fig.add_trace(go.Surface(
    x=xx, y=yy, z=zz,
    colorscale='Greys',
    opacity=0.7,
    showscale=False,
    name='Hyperplane'
))

# config plot
fig.update_layout(
    title='clasification of species (Perceptron-based)',
    scene=dict(
        xaxis_title='Sepal Length',
        yaxis_title='Sepal Width',
        zaxis_title='Petal Length'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# plot
fig.show()


### MLP Classification through binary partiction of R<sup>3</sup> space

In [ ]:
# use same iris data loaded from before


# 3 features for 3D plotting are in X_3d and labels in y_binary

# scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_3d)


# train
# Using 2 hidden layers to allow for a curved boundary
clf_mlp = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', max_iter=2000, random_state=42)
clf_mlp.fit(X_scaled, y_binary)

# create a 3D meshgrid for decision volume
# We generate the grid based on the original unscaled data so the plot axes remain readable
grid_resolution = 30 
x_min, x_max = X_3d[:, 0].min() - 0.5, X_3d[:, 0].max() + 0.5
y_min, y_max = X_3d[:, 1].min() - 0.5, X_3d[:, 1].max() + 0.5
z_min, z_max = X_3d[:, 2].min() - 0.5, X_3d[:, 2].max() + 0.5

xx, yy, zz = np.meshgrid(
    np.linspace(x_min, x_max, grid_resolution),
    np.linspace(y_min, y_max, grid_resolution),
    np.linspace(z_min, z_max, grid_resolution)
)

# flat  3D grid, scale, and predict probabilities
grid_points = np.c_[xx.ravel(), yy.ravel(), zz.ravel()]
grid_points_scaled = scaler.transform(grid_points)

# Get the probability that a point belongs to Class 1 (Versicolor)
probs = clf_mlp.predict_proba(grid_points_scaled)[:, 1]
probs = probs.reshape(xx.shape)

# plot
fig = go.Figure()

# Setosa points
fig.add_trace(go.Scatter3d(
    x=X_3d[y_binary == 0, 0],
    y=X_3d[y_binary == 0, 1],
    z=X_3d[y_binary == 0, 2],
    mode='markers',
    marker=dict(size=6, color='pink', opacity=0.8),
    name='Setosa'
))

# Versicolor points
fig.add_trace(go.Scatter3d(
    x=X_3d[y_binary == 1, 0],
    y=X_3d[y_binary == 1, 1],
    z=X_3d[y_binary == 1, 2],
    mode='markers',
    marker=dict(size=6, color='green', opacity=0.8),
    name='Versicolor'
))

# decision boundary as  Isosurface where probability = .5
fig.add_trace(go.Isosurface(
    x=xx.flatten(), 
    y=yy.flatten(), 
    z=zz.flatten(),
    value=probs.flatten(),
    isomin=0.5, 
    isomax=0.5,           # draw surface at 50% probability threshold
    opacity=0.6,
    surface=dict(count=1),
    colorscale='Greys',
    caps=dict(x_show=False, y_show=False, z_show=False),
    showscale=False,
    name='Decision Boundary'
))

# con fig plot
fig.update_layout(
    title='Class of species (MLLP-based)',
    scene=dict(
        xaxis_title='Sepal Length',
        yaxis_title='Sepal Width',
        zaxis_title='Petal Length'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# show
fig.show()

In [22]:
# data: iris
X_full, y_full = iris.frame.iloc[:, :4].values, iris.frame['target'].values
target_names = iris.target_names

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_full, test_size=0.2, random_state=42)

# Convert to MLX arrays (both inputs and targets)
X_train_mx = mx.array(X_train, dtype=mx.float32)
y_train_mx = mx.array(y_train, dtype=mx.int32)
X_test_mx = mx.array(X_test, dtype=mx.float32)

# 2. Define MLX Architecture
class IrisMLP(mnn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = mnn.Linear(4, 8)
        self.output = mnn.Linear(8, 3)

    def __call__(self, x):
        # We output RAW LOGITS here for training stability.
        # We will apply softmax later for the final predictions.
        x = mnn.relu(self.hidden(x))
        logits = self.output(x)
        return logits

model = IrisMLP()
mx.eval(model.parameters())

# optimizer and loss
optimizer = optim.SGD(learning_rate=0.01)

def loss_fn(model, X, y):
    logits = model(X)
    # MLX's cross_entropy handles the softmax internally for stability
    loss = mx.mean(mnn.losses.cross_entropy(logits, y))
    return loss

# loss and gradients with mlx
loss_and_grad_fn = mnn.value_and_grad(model, loss_fn)

# train loop
epochs = 400
print("Starting Training...")

for epoch in range(epochs):
    # loss and gradients
    loss, grads = loss_and_grad_fn(model, X_train_mx, y_train_mx)
    
    # update model parameters
    optimizer.update(model, grads)
    
    # look more on this
    mx.eval(model.parameters(), optimizer.state)
    
    # progress every 50 epochs
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d} | Loss: {loss.item():.4f}")

# output
print("\nTraining Complete. Evaluating on Test Set...")

# softmax gets us probs
test_logits = model(X_test_mx)
test_probs = mnn.softmax(test_logits, axis=1)

probs_np = np.array(test_probs[:5])
true_labels_idx = y_test[:5]

results = []
for i in range(5):
    results.append({
        "True Label": target_names[true_labels_idx[i]].capitalize(),
        "Head 0 (Setosa)": f"{probs_np[i, 0]:.4f}",
        "Head 1 (Versicolor)": f"{probs_np[i, 1]:.4f}",
        "Head 2 (Virginica)": f"{probs_np[i, 2]:.4f}"
    })

print("\nOutput Mapping: Head 0 -> Setosa | Head 1 -> Versicolor | Head 2 -> Virginica\n")
print(pd.DataFrame(results).to_string(index=False))

Starting Training...
Epoch  50 | Loss: 1.0930
Epoch 100 | Loss: 0.9521
Epoch 150 | Loss: 0.8265
Epoch 200 | Loss: 0.7128
Epoch 250 | Loss: 0.6204
Epoch 300 | Loss: 0.5508
Epoch 350 | Loss: 0.4989
Epoch 400 | Loss: 0.4594

Training Complete. Evaluating on Test Set...

Output Mapping: Head 0 -> Setosa | Head 1 -> Versicolor | Head 2 -> Virginica

True Label Head 0 (Setosa) Head 1 (Versicolor) Head 2 (Virginica)
Versicolor          0.1401              0.4671             0.3927
    Setosa          0.9177              0.0291             0.0532
 Virginica          0.0175              0.1627             0.8198
Versicolor          0.1197              0.4620             0.4183
Versicolor          0.0992              0.4317             0.4692


In [24]:
# data
X_full = df.iloc[:, :4].values
y_full = df['target'].values

# scale split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_full, test_size=0.2, random_state=42)

# PyTorch tensors
X_tensor = torch.tensor(X_test, dtype=torch.float32)

# MLP archi
class IrisMLP(nn.Module):
    def __init__(self):
        super(IrisMLP, self).__init__()
        self.hidden = nn.Linear(4, 8)  # 4 inputs to 8 hidden nodes
        self.output = nn.Linear(8, 3)  # 8 hidden nodes to 3 output heads
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1) # Forces 0 to 1 activation across the 3 heads

    def forward(self, x):
        x = self.relu(self.hidden(x))
        logits = self.output(x)
        probs = self.softmax(logits)
        return probs

model = IrisMLP()

# output, pending improve output structure as in MLX model
with torch.no_grad():
    predictions = model(X_tensor)
    
print("Sample Output from the 3 Heads (Values between 0 and 1):")
print(np.round(predictions[:5].numpy(), 4))

Sample Output from the 3 Heads (Values between 0 and 1):
[[0.5342 0.1986 0.2672]
 [0.4827 0.1879 0.3294]
 [0.8159 0.0453 0.1388]
 [0.5129 0.2203 0.2669]
 [0.6066 0.1588 0.2346]]


In [25]:
# data
iris = load_iris(as_frame=True)
df = iris.frame

# variable setting
X_reg = df[['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)']].values
y_reg = df['petal width (cm)'].values

# data split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# feature scaleing
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reg)
X_test_scaled = scaler.transform(X_test_reg)

# def and train mlp
# 3 inputs -> 16 hidden nodes -> 8 hidden nodes -> 1 linear output
mlp_regressor = MLPRegressor(
    hidden_layer_sizes=(16, 8), 
    activation='relu',          # Non-linear activation for hidden layers
    solver='sgd',              
    max_iter=1500,              # Epochs
    random_state=42
)

mlp_regressor.fit(X_train_scaled, y_train_reg)

# eval
y_pred_reg = mlp_regressor.predict(X_test_scaled)
mse = mean_squared_error(y_test_reg, y_pred_reg)

print(f"Mean Squared Error: {mse:.4f}")

# first 5 predictions vs true values
comparison = pd.DataFrame({'Actual': y_test_reg[:5], 'Predicted': y_pred_reg[:5]})
print("\nFirst 5 Predictions:")
print(comparison.round(3))

Mean Squared Error: 0.0699

First 5 Predictions:
   Actual  Predicted
0     1.2      1.631
1     0.3      0.312
2     2.3      2.707
3     1.5      1.468
4     1.4      1.874
